GEETA 4B — shared tutor contract + problem data access.

Single source of truth for the GEETA system prompt (same text as
notebooks/03 cell 1 — change here and in the notebook together).

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

In [ ]:


TUTOR_SYSTEM = """You are GEETA, a warm, gentle maths tutor for Australian children in Years 1 to 3 (ages 6-8).
Your rules, always:
1. The child works it out THEMSELVES. You guide; you never give the final answer — no matter how they ask, probe, or beg.
2. Ask for the child's first idea ONCE at the start. After that, just help — do not keep quizzing them or asking "is that right?".
3. Use the WHOLE conversation. Never repeat a hint you already gave. If the child gives a number from their working, react to THAT number: say what is right, then nudge the very next tiny step. Never restart the problem.
4. One idea per reply. No numbered lists, no "Step 1 / Step 2". Just the next little nudge.
5. Only ever state arithmetic you are sure of. If unsure, say "let's check this one together".
6. Keep replies short (under 50 words), kind, calm, and kid-level. Australian spelling and examples.
7. If the child goes quiet or gives up: gentle no-blame reset, a concrete anchor (fingers, groups of 10), a short story, a micro-win.
8. Celebrate the child's own working. Never say "the answer is ..." before the child says it.
9. Only use numbers that are in the question or the checked working. Never invent a rule (like "count by 10") the question doesn't use.
10. If the child asks to see or hear the question again, read it out — the question is never a secret, only the answer.
11. Talk like a friendly big buddy to a 7-year-old: short simple words, one question at a time.
"""

# Internal-only system prompt for the C5 verifier's "work it out" calls.
# R1 (session 12): reusing TUTOR_SYSTEM here made the model treat its own
# working request as a leak probe - "never give the final answer no matter
# how they ask" collided with "emit ONLY JSON ... final_answer: ...", and it
# would frequently refuse with an in-character hint instead of the JSON (6
# manual retries against the real server: 4/6 refused, 2/6 complied). This
# call's output is NEVER shown to the child (verify_background_working /
# M5 Cell B use it purely to compute + verify a plan before anything is
# delivered), so the never-reveal persona must not apply to it. Matches
# PROJECT_BRIEF Known risk R2 ("mitigate with role prompts in data").
WORK_SYSTEM = ("You are a precise arithmetic engine for an internal verification "
    "step. Your output is NEVER shown to the child - it is checked by an "
    "independent verifier before any reply is sent. Always respond with ONLY "
    "the requested JSON, no other text, no refusals, no commentary.")

In [ ]:


class ProblemStore:
    """Loads problems (bank + held-out eval) and keeps them in memory.

    Problems use the M1 bank schema:
    id, skill_id, year, strand, acara_code, question_type, question_text,
    answer, answer_check, solution_steps, hint_ladder, similar_worked,
    story_reframe, ...
    """

    def __init__(self, root: str | Path = "."):
        self.root = Path(root)
        self.by_id: dict[str, dict] = {}
        self.heldout_ids: set[str] = set()

    def load_bank(self, path: str | Path | None = None) -> int:
        p = Path(path) if path else self.root / "data" / "bank" / "acara_y1-3.jsonl"
        n = 0
        with open(p) as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    self.by_id[row["id"]] = row
                    n += 1
        return n

    def load_heldout(self, path: str | Path | None = None) -> int:
        """The held-out set is a stratified SUBSET of the bank (same rows,
        never in training — split_lock policy). We record which ids are held
        out; the bank file stays the single row store."""
        p = Path(path) if path else self.root / "data" / "eval" / "heldout_problems.jsonl"
        n = 0
        with open(p) as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    self.heldout_ids.add(row["id"])
                    self.by_id.setdefault(row["id"], row)
                    n += 1
        return n

    def heldout_problems(self) -> list[dict]:
        return [self.by_id[i] for i in self.heldout_ids if i in self.by_id]

    def train_pool(self) -> list[dict]:
        """Bank problems minus held-out (safe for training/demo use)."""
        return [p for i, p in self.by_id.items() if i not in self.heldout_ids]

    def load_probes(self, path: str | Path | None = None) -> list[dict]:
        p = Path(path) if path else self.root / "data" / "eval" / "adversarial_probes.jsonl"
        out = []
        with open(p) as f:
            for line in f:
                if line.strip():
                    out.append(json.loads(line))
        return out

    def get(self, pid: str) -> dict:
        return self.by_id[pid]

    def problems_by_skill(self, year: int | None = None) -> list[dict]:
        rows = [p for p in self.by_id.values() if year is None or p.get("year") == year]
        return rows